# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Hammad-vibes/FlyRank-Internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

One row = one content item (within one client). Features come only from 1-15 March (h1). The label uses 16-31 March (h2) and is never a feature. The five engineered features are impressions, clicks, CTR, average position and active days. There is no categorical feature: `client_hash_id` is used only to group the split, never as a model input. Missing values: `ctr_h1` is empty when impressions are 0, so those rows are dropped; items with fewer than 5 h1 clicks are dropped; missing h2 clicks are filled with 0 but only for the label.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import duckdb, numpy as np, pandas as pd
from google.colab import userdata

con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{userdata.get('HF_TOKEN')}')")
MAR = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet"
con.execute(f"CREATE VIEW mar AS SELECT * FROM read_parquet('{MAR}')")

frame = con.sql("""
SELECT client_hash_id, content_hash_id,
  SUM(gsc_impressions)  FILTER (WHERE report_date <= DATE '2026-03-15') AS impressions_h1,
  SUM(gsc_clicks)       FILTER (WHERE report_date <= DATE '2026-03-15') AS clicks_h1,
  AVG(gsc_avg_position) FILTER (WHERE report_date <= DATE '2026-03-15') AS avg_position_h1,
  COUNT(*) FILTER (WHERE report_date <= DATE '2026-03-15' AND gsc_impressions > 0) AS active_days_h1,
  SUM(gsc_clicks)       FILTER (WHERE report_date >  DATE '2026-03-15') AS clicks_h2
FROM mar
WHERE gsc_data_available IS TRUE
GROUP BY 1, 2
""").df()

df = frame.copy()
df["h2_missing"] = df["clicks_h2"].isna()
df["clicks_h2"] = df["clicks_h2"].fillna(0)
df["ctr_h1"] = df["clicks_h1"] / df["impressions_h1"].replace(0, np.nan)
FEATURES = ["impressions_h1", "clicks_h1", "ctr_h1", "avg_position_h1", "active_days_h1"]
df = df[df["clicks_h1"] >= 5].dropna(subset=FEATURES).copy()
df["label"] = (df["clicks_h2"] < 0.8 * df["clicks_h1"]).astype(int)
df = df.reset_index(drop=True)

print("rows:", len(df), "| clients:", df["client_hash_id"].nunique(), "| decline rate:", round(df["label"].mean(), 3))
display(df[FEATURES + ["label"]].describe().round(2))
print("missing values per feature:"); print(df[FEATURES].isna().sum().to_string())


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

rows: 17431 | clients: 29 | decline rate: 0.442


,impressions_h1,clicks_h1,ctr_h1,avg_position_h1,active_days_h1,label
count,17431.00,17431.00,17431.00,17431.00,17431.00,17431.00
mean,4481.57,18.38,0.01,8.80,14.40,0.44
std,6236.15,35.18,0.01,8.96,1.74,0.50
min,7.00,5.00,0.00,0.21,2.00,0.00
25%,1373.00,7.00,0.00,3.27,15.00,0.00
50%,2626.00,10.00,0.00,4.97,15.00,0.00
75%,5151.50,19.00,0.01,10.20,15.00,1.00
max,161575.00,2395.00,0.71,56.00,15.00,1.00


missing values per feature:
impressions_h1     0
clicks_h1          0
ctr_h1             0
avg_position_h1    0
active_days_h1     0


## 2. Feature notes (meaning, missing, categorical, available-when?)

Each feature is built from days 1-15 March only, so each one exists at the decision moment (end of 15 March). The table in the code cell below says it feature by feature, and the query checks the date windows behind it.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
notes = pd.DataFrame([
    ["impressions_h1",  "total Search impressions, 1-15 Mar", "rows with no h1 data do not exist", "yes, summed before 16 Mar"],
    ["clicks_h1",       "total clicks, 1-15 Mar",             "min 5 clicks required",           "yes, summed before 16 Mar"],
    ["ctr_h1",          "clicks_h1 / impressions_h1",          "empty if impressions = 0 (row dropped)", "yes, both parts are h1"],
    ["avg_position_h1", "mean of daily average position, 1-15 Mar", "rows missing it dropped", "yes, daily values before 16 Mar"],
    ["active_days_h1",  "days with impressions > 0, 1-15 Mar", "0 if none (cannot happen after filter)", "yes, counted before 16 Mar"],
], columns=["feature", "meaning", "missing handling", "available before 16 Mar?"])
display(notes)

# proof: the feature window stops at 15 March, the label window starts at 16 March
con.sql("""
SELECT MIN(report_date) AS first_day_used_for_features,
       MAX(report_date) FILTER (WHERE report_date <= DATE '2026-03-15') AS last_day_used_for_features,
       MIN(report_date) FILTER (WHERE report_date >  DATE '2026-03-15') AS first_day_used_for_label
FROM mar WHERE gsc_data_available IS TRUE
""").show()
assert not ({"label", "clicks_h2", "h2_missing"} & set(FEATURES)), "label/future column in FEATURES"


,feature,meaning,missing handling,available before 16 Mar?
0,impressions_h1,"total Search impressions, 1-15 Mar",rows with no h1 data do not exist,"yes, summed before 16 Mar"
1,clicks_h1,"total clicks, 1-15 Mar",min 5 clicks required,"yes, summed before 16 Mar"
2,ctr_h1,clicks_h1 / impressions_h1,empty if impressions = 0 (row dropped),"yes, both parts are h1"
3,avg_position_h1,"mean of daily average position, 1-15 Mar",rows missing it dropped,"yes, daily values before 16 Mar"
4,active_days_h1,"days with impressions > 0, 1-15 Mar",0 if none (cannot happen after filter),"yes, counted before 16 Mar"


┌─────────────────────────────┬────────────────────────────┬──────────────────────────┐
│ first_day_used_for_features │ last_day_used_for_features │ first_day_used_for_label │
│            date             │            date            │           date           │
├─────────────────────────────┼────────────────────────────┼──────────────────────────┤
│ 2026-03-01                  │ 2026-03-15                 │ 2026-03-16               │
└─────────────────────────────┴────────────────────────────┴──────────────────────────┘



## 3. The leakage hunt

I attack my own features with four tests: (1) does any single feature look suspiciously strong, (2) does a shuffled label still score above 0.5, (3) what happens when I add a label-derived column on purpose, (4) do the click-based features that also sit inside the label formula carry the score? Scores use a client-grouped split. In my earlier runs (Week 3 and Week 6) the honest score was about 0.57-0.59, the shuffled-label score was close to 0.5, and the deliberate leak pushed the score to about 0.996. Read the outputs below for this run. These tests find leakage I thought to look for; they do not prove there is none.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.model_selection import GroupKFold
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler, FunctionTransformer
from sklearn.metrics import roc_auc_score

y = df["label"].values
groups = df["client_hash_id"].values
folds = list(GroupKFold(n_splits=5).split(df, y, groups))

def oof_auc(cols, yv=None):
    yv = y if yv is None else yv
    X = df[cols]
    s = np.zeros(len(df))
    for tr, te in folds:
        m = make_pipeline(FunctionTransformer(np.log1p), StandardScaler(), LogisticRegression(max_iter=1000))
        m.fit(X.iloc[tr], yv[tr]); s[te] = m.predict_proba(X.iloc[te])[:, 1]
    return roc_auc_score(yv, s)

# 1) one feature at a time (direction-free AUC)
uni = {}
for f in FEATURES:
    a = roc_auc_score(y, df[f]); uni[f] = max(a, 1 - a)
print("1) univariate ROC-AUC per feature (0.5 = no signal):"); print(pd.Series(uni).round(3).to_string())

# 2) shuffled labels should score about 0.5
y_shuf = np.random.RandomState(0).permutation(y)
print("\n2) shuffled-label ROC-AUC:", round(oof_auc(FEATURES, yv=y_shuf), 3))

# 3) deliberate leak: add the future quantity the label is built from, then remove it
honest = oof_auc(FEATURES)
df["leak_clicks_h2"] = df["clicks_h2"]
leaky = oof_auc(FEATURES + ["leak_clicks_h2"])
df = df.drop(columns="leak_clicks_h2")
print("\n3) honest:", round(honest, 3), "| with leak:", round(leaky, 3), "| after removing leak:", round(oof_auc(FEATURES), 3), "<- number I keep")

# 4) drop the click features that also appear in the label formula
print("\n4) without clicks_h1 and ctr_h1:", round(oof_auc(["impressions_h1", "avg_position_h1", "active_days_h1"]), 3), "| all five:", round(honest, 3))

1) univariate ROC-AUC per feature (0.5 = no signal):
impressions_h1     0.549
clicks_h1          0.542
ctr_h1             0.511
avg_position_h1    0.536
active_days_h1     0.539

2) shuffled-label ROC-AUC: 0.492

3) honest: 0.586 | with leak: 1.0 | after removing leak: 0.586 <- number I keep

4) without clicks_h1 and ctr_h1: 0.586 | all five: 0.586


## 4. What I excluded and why

- **All `ga4_*`, `sessions_*`, `ai_*`, `scroll_events` columns:** other lanes, and GA4 data exists on only about 4% of March rows.
- **Anything dated 16-31 March as a feature:** it is future information at the decision moment.
- **`clicks_h2` and the label:** they are the outcome. The leak test above shows what happens if they slip in.
- **`h2_missing`:** built from future data, kept only for the audit.
- **`client_hash_id` and `content_hash_id` as model inputs:** identifiers would let the model memorise clients instead of learning patterns; they are used only for grouping.
- **June 2026 (final month):** sealed test month, not touched while developing.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
excluded = {
    "ga4_*, sessions_*, ai_*, scroll_events": "other lanes; GA4 rare in March",
    "any column dated 16-31 March (as feature)": "future information",
    "clicks_h2 / label": "the outcome itself",
    "h2_missing": "built from future data, audit only",
    "client_hash_id, content_hash_id (as inputs)": "identifiers; used only for grouping",
    "June 2026": "sealed test month",
}
for k, v in excluded.items():
    print(f"{k:45} -> {v}")

ga4_*, sessions_*, ai_*, scroll_events        -> other lanes; GA4 rare in March
any column dated 16-31 March (as feature)     -> future information
clicks_h2 / label                             -> the outcome itself
h2_missing                                    -> built from future data, audit only
client_hash_id, content_hash_id (as inputs)   -> identifiers; used only for grouping
June 2026                                     -> sealed test month


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.